In [ ]:
# %pip install sense_hat

In [ ]:
import time, random
from sense_hat import SenseHat

In [ ]:
sense = SenseHat()

In [ ]:
#defining the colors
snake_color = (0, 255, 0) #green
food_color = (255, 0, 0) #red
clear = (0, 0, 0) #black

In [ ]:
sense.show_message("3 2 1 go!", text_colour=snake_color)

In [ ]:
# initial configurations
snake_position = [[2, 2]] #initial position of the snake
moving_direction = [0, 1] #vector that moves the snake to the right
snake_length = 1 #initial length of the snake
food_position = [random.randint(0,7), random.randint(0,7)] #random position for the food #storing food in 2d ccoordinate

background_color = [clear] * 64 #background color is black #pixels are stored in 1d array

In [ ]:
#method to set the direction
#store direction as a vector, to easily add it to the snake's position to move it

def set_directions(direction):
    global moving_direction
    if direction == "up":
        moving_direction = [0, 1]
    elif direction == "down":
        moving_direction = [0, -1]
    elif direction == "left":
        moving_direction = [-1, 0]
    elif direction == "right":
        moving_direction = [1, 0]

In [ ]:
# generated with claude sonnet 4.6
# prompt: " generate me an explosion effect starting in the middle, moving outwards to the edges in white and blue ccolors 
# on my rasperry pi sense hat snake game programmed with python"
def explosion_effect(sense):
    # Define explosion colors
    Y = (255, 255, 0)  # Yellow
    B = (0, 100, 255)    # Blue
    O = (0, 0, 0)        # Off/Black

    # Center of the 8x8 grid
    center = (3, 3)

    # Clear screen
    sense.clear()

    # Expand rings outward from center
    for radius in range(5):  # 0 to 4 covers the whole grid
        pixels = [[0, 0, 0]] * 64  # Start with black canvas

        for x in range(8):
            for y in range(8):
                dist = max(abs(x - center[0]), abs(y - center[1]))  # Chebyshev distance

                if dist == radius:
                    pixels[y * 8 + x] = list(Y)  # Current ring = Yellow
                elif dist == radius - 1:
                    pixels[y * 8 + x] = list(B)  # Previous ring = Blue (fading)

        sense.set_pixels(pixels)
        time.sleep(0.15)  # Speed of explosion

    time.sleep(0.2)
    sense.clear()

In [ ]:
#game loop
while True:

    #initial setting
    pixels = [clear] * 64 #clear the background

    #snake body
    for pos in snake_position:
        pixels[pos[1] * 8 + pos[0]] = snake_color #formula: y * rowSize + x   #bc of 2d and 1d array


    #Random Position of the red food on the game board
    # formula: y * rowSize + x   #bc of 2d and 1d array conversion
    pixels[food_position[1] * 8 + food_position[0]] = food_color
    sense.set_pixels(pixels) #display the food

    for move in sense.stick.get_events(): 
        if move.action == "pressed":
            set_directions(move.direction)

    #movement of the snake
    snake_position.insert(0, [snake_position[0][0] + moving_direction[0], snake_position[0][1] + moving_direction[1]]) #move the snake by adding the direction vector to the head of the snake


    #Collision with the wall 
    if snake_position[0][0] < 0 or snake_position[0][0] > 7 or snake_position[0][1] < 0 or snake_position[0][1] > 7: #check if the head of the snake collides with the wall
        explosion_effect(sense)
        sense.show_message("Game Over!", text_colour=food_color) #display game over message
        break

    #
    if snake_position[0] == food_position:
        food_position = []
        while food_position == []:
            food_position = [random.randint(0,7), random.randint(0,7)] # task (f)
            if food_position in snake_position:
                food_position = []
        snake_length += 1
    
    elif snake_position[0] in snake_position[1:]: #check if the head of the snake collides with its body 
        explosion_effect(sense)
        sense.show_message("Game Over!", text_colour=food_color) #display game over message
        break


    else:
        while len(snake_position) > snake_length:
            snake_position.pop()

    time.sleep(0.5) #wait 500ms as specified in the exercise task (g)